# 04 — Group Relative Policy Optimization (GRPO) on CPU

**GRPO** is reinforcement learning with a *verifiable reward*: there are no target answers and no preference
pairs. For each prompt the model samples a group of answers, a reward function scores them, and the model moves
towards the answers that scored above their group's average. It is how reasoning models are trained on math and
code, where the answer can be checked.

Same booking task. The reward (`booking_reward` in `scripts/booking_data.py`):

* 1.0 for bare JSON that exactly matches
* otherwise 0.6 × the share of correct fields, + 0.2 if it is bare JSON (no markdown code block)
* 0 if no JSON can be parsed

Partial credit matters: the base model's answers are never exactly right, so without it every answer in a group
would score 0 and there would be nothing to learn from. The job samples 4 answers for each of 4 requests per step
(no vLLM, no KL term so no reference model), for 40 steps. Expected: ~10 minutes, most of it sampling on CPU.

In [ ]:
%pip install -q "sagemaker==3.21.0"

## Setup

`Lab` (in `sm_helpers.py`) holds the steps notebook 01 spells out: the SageMaker session, execution role and S3
bucket, MLflow, and a CPU training job. Two settings:

* `CAPACITY`: how the job gets its instance: `"fallback"` (the first of several CPU types with capacity; the
  default), `"on-demand"` (exactly `ml.c7i.4xlarge`) or `"spot"` (cheaper, may wait for capacity). Notebook 01
  compares the three.
* `MLFLOW_APP_ARN`: where the job logs its metrics (see *Track in MLflow* below).

In [ ]:
import sys

sys.path.insert(0, "scripts")
from sm_helpers import Lab

# MLflow: "" = reuse the account's MLflow App (or create one: several minutes, first time only);
# "arn:aws:sagemaker:..." = use that app; None = off. Jobs run the same either way.
MLFLOW_APP_ARN = ""
lab = Lab(mlflow_app_arn=MLFLOW_APP_ARN)
CAPACITY = "fallback"   # "on-demand" | "spot" | "fallback" (compared side by side in 01-sft.ipynb)

In [ ]:
import booking_data

train, _ = booking_data.make_splits()
target = train[0]["target"]
for answer in (target, "```json\n" + target + "\n```", target.replace("Sunday", "sunday"), "sorry, no idea"):
    print(f"{booking_data.booking_reward(answer, target):.2f}  {answer!r}")

## Train

One job runs `scripts/train_grpo.py` with TRL's `GRPOTrainer`. Each step takes 4 requests, samples 4 answers for
each (temperature 1.0, so they differ), scores all 16 with the reward function, and updates the adapter towards
the answers that scored above their group's average. Sampling dominates the ~13 minutes: generation on CPU is
slow, which is why GPU GRPO recipes use vLLM for it. The wait cell prints status changes; if the kernel restarts,
re-run setup and the wait cell.

In [ ]:
job = lab.submit("grpo", "train_grpo.py", {"steps": 40}, capacity=CAPACITY, requirements="requirements-trl.txt")

In [ ]:
job = globals().get("job") or lab.latest("grpo")   # after a kernel restart
desc = lab.wait(job)

## Results

In [ ]:
m = lab.metrics(job)
print(f"{'':10} {'exact JSON':>10} {'mean reward':>12} " + " ".join(f"{f:>10}" for f in ("name", "party_size", "day", "time")))
for which in ("base", "finetuned"):
    r = m[which]
    print(f"{which:10} {r['exact_match']:10.0%} {r['mean_reward']:12.2f} "
          + " ".join(f"{r['acc_' + f]:10.0%}" for f in ("name", "party_size", "day", "time")))
curve = [p["reward"] for p in m["reward_curve"] if p["reward"] is not None]
print("\nreward per step (training samples):")
print(" ".join(f"{r:.2f}" for r in curve))
ex = m["examples"][0]
print(f"\nbase:      {ex['base']!r}\nfinetuned: {ex['finetuned']!r}\ntarget:    {ex['target']}")

### How to read the results

| Row | Model |
|---|---|
| `base` | Qwen3-0.6B before training |
| `finetuned` | the same model + the LoRA adapter trained with GRPO for 40 steps |

| Column | What it measures |
|---|---|
| exact JSON | on 30 held-out requests, the share of answers that are bare JSON and equal the target exactly |
| mean reward | the average of the same reward function used in training (`booking_reward`) on those answers: 1.0 = exact; partial credit for correct fields; 0 = no JSON |
| name, party_size, day, time | per field: the share of answers where that field is right (the JSON may still be wrapped in a code block) |

"reward per step" is the average reward of the answers the model **sampled during training** at each step (16
answers per step: 4 requests × 4 samples). It is noisy because each step sees different requests, but it should
trend upwards.

**What happens.** The reward goes up (e.g. ~0.45 → ~0.8) and the name, party size and day fields reach ~100%,
but exact match only reaches ~40%. Look at the `time` column: the base model writes "7pm" style times and almost
never samples "19:00", so GRPO rarely sees a high-reward example with a 24-hour time to reinforce.

## Track in MLflow

Each job also logs its parameters, the training loss per step, and the base and fine-tuned scores to MLflow,
one run per job in the experiment `cpu-fine-tuning-techniques`. Select runs and **Compare** to put the techniques
side by side. The link below is valid for a few minutes; re-run the cell for a new one.

In [ ]:
# Compare runs in MLflow: experiment "cpu-fine-tuning-techniques", one run per job (named after the job)
print(lab.mlflow_url() or "MLflow is off")

## Takeaway

GRPO only reinforces what the model already samples sometimes. It quickly drops the markdown code block and
fixes the weekday (both appear among its samples), but learns slowly what it almost never does on its own (the
24-hour time), so exact match lags behind SFT. That is the trade-off: RL needs no labelled answers, only a way to
score them, but it cannot easily discover new behaviors. Starting GRPO from an SFT model is the usual recipe; it
is how reasoning models are trained on math and code, where answers can be checked automatically.